# NormLens Clause Classifier Training
Fine-tunes Legal-BERT on synthetic clause data (+ optionally CUAD + LEDGAR).
On Colab GPU: ~5-10 minutes.

In [ ]:
!pip install torch transformers "datasets<4.0" evaluate accelerate sentence-transformers joblib -q

In [ ]:
EPOCHS = 5
BATCH_SIZE = 16
USE_SYNTHETIC = False
LR = 2e-5
OUTPUT_DIR = "/content/clause_classifier"

In [ ]:
import json, logging, os, numpy as np
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("normlens")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

BASE_MODEL = "nlpaueb/legal-bert-base-uncased"

LABELS = [
    "Termination", "Payment Terms", "Liability", "Limitation of Liability",
    "Confidentiality", "Non-Compete", "Intellectual Property", "Indemnification",
    "Assignment", "Governing Law", "Arbitration", "Insurance", "Data Protection",
    "Data Ownership", "Security Obligations", "Force Majeure", "Warranty",
    "Representations and Warranties", "Entire Agreement", "Amendments", "Notices",
    "Severability", "Waiver", "Survival", "Counterparts", "Definitions",
    "Expenses", "Publicity", "Subcontracting", "No Waiver", "Further Assurances",
    "No Third Party Beneficiaries",
]

CUAD_MAP = {
    "notice period for termination": "Termination",
    "termination for convenience": "Termination",
    "termination for cause": "Termination",
    "payment schedule": "Payment Terms",
    "late payment": "Payment Terms",
    "liability": "Liability",
    "limitation of liability": "Limitation of Liability",
    "caps on liability": "Limitation of Liability",
    "indemnification": "Indemnification",
    "non-compete": "Non-Compete",
    "confidential information": "Confidentiality",
    "confidentiality": "Confidentiality",
    "intellectual property": "Intellectual Property",
    "ownership of ip": "Intellectual Property",
    "assignment": "Assignment",
    "governing law": "Governing Law",
    "jurisdiction": "Governing Law",
    "arbitration": "Arbitration",
    "insurance": "Insurance",
    "data protection": "Data Protection",
    "force majeure": "Force Majeure",
    "warranty": "Warranty",
    "representations and warranties": "Representations and Warranties",
    "entire agreement": "Entire Agreement",
    "amendments": "Amendments",
    "notice": "Notices",
    "severability": "Severability",
    "waiver": "Waiver",
    "survival": "Survival",
    "counterparts": "Counterparts",
    "definitions": "Definitions",
    "expenses": "Expenses",
    "publicity": "Publicity",
    "subcontracting": "Subcontracting",
    "non-waiver": "No Waiver",
    "further assurances": "Further Assurances",
    "third party beneficiaries": "No Third Party Beneficiaries",
    "renewal": "Payment Terms",
    "license grant": "Intellectual Property",
    "non-disclosure": "Confidentiality",
    "covenant": "Warranty",
    "audit": "Security Obligations",
    "security": "Security Obligations",
    "data ownership": "Data Ownership",
    "data use": "Data Ownership",
}

LEDGAR_MAP = {
    "Termination": "Termination", "Payment Terms": "Payment Terms",
    "Liability": "Liability", "Limitation of Liability": "Limitation of Liability",
    "Confidentiality": "Confidentiality", "Non-Compete": "Non-Compete",
    "Intellectual Property": "Intellectual Property", "Indemnification": "Indemnification",
    "Assignment": "Assignment", "Governing Law": "Governing Law",
    "Arbitration": "Arbitration", "Insurance": "Insurance",
    "Data Protection": "Data Protection", "Force Majeure": "Force Majeure",
    "Warranty": "Warranty", "Amendment": "Amendments", "Notices": "Notices",
    "Waiver": "Waiver", "Survival": "Survival", "Definitions": "Definitions",
    "Expenses": "Expenses", "Publicity": "Publicity",
}

TEMPLATES = {
    "Termination": [
        "This agreement may be terminated by either party upon {n} days written notice.",
        "Either party may terminate this agreement for cause upon material breach.",
        "In the event of termination, all rights and obligations shall cease.",
    ],
    "Payment Terms": [
        "All invoices shall be paid within {n} days of receipt.",
        "Payment shall be made via wire transfer within {n} calendar days.",
        "A late fee of {pct}% per month shall apply to overdue amounts.",
    ],
    "Liability": [
        "Neither party shall be liable for any indirect or consequential damages.",
        "The total liability of either party shall not exceed ${amount}.",
        "Nothing in this agreement shall limit liability for fraud or gross negligence.",
    ],
    "Limitation of Liability": [
        "In no event shall either party's aggregate liability exceed ${amount}.",
        "The limitation of liability set forth herein shall not apply to indemnification obligations.",
        "Exclusive remedy for any claim is limited to direct damages not exceeding ${amount}.",
    ],
    "Confidentiality": [
        "The receiving party shall maintain confidentiality for a period of {n} years.",
        "Confidential information shall not be disclosed to third parties without prior written consent.",
        "This non-disclosure obligation shall survive termination for {n} years.",
    ],
    "Non-Compete": [
        "The party shall not engage in any competing business for {n} months post-termination.",
        "For a period of {n} months, the consultant shall not solicit any clients.",
    ],
    "Intellectual Property": [
        "All intellectual property created during the term shall be owned by the company.",
        "The contractor hereby assigns all rights, title, and interest in the work product.",
        "Pre-existing IP shall remain the property of the originating party.",
    ],
    "Indemnification": [
        "Provider shall indemnify and hold harmless customer from all third-party claims.",
        "Each party agrees to defend the other against claims arising from IP infringement.",
    ],
    "Assignment": [
        "Neither party may assign this agreement without the other's prior written consent.",
        "This agreement shall be binding upon and inure to the benefit of permitted assigns.",
    ],
    "Governing Law": [
        "This agreement shall be governed by the laws of the State of New York.",
        "The parties submit to the exclusive jurisdiction of the federal courts in Delaware.",
    ],
    "Arbitration": [
        "Any dispute arising out of this agreement shall be resolved by binding arbitration.",
        "The arbitration shall be conducted in accordance with AAA rules.",
    ],
    "Insurance": [
        "Provider shall maintain commercial general liability insurance of ${amount} per occurrence.",
        "Upon request, provider shall furnish certificates of insurance evidencing coverage.",
    ],
    "Data Protection": [
        "Both parties shall comply with all applicable data protection laws including GDPR.",
        "Personal data shall only be processed in accordance with the data processing agreement.",
    ],
    "Force Majeure": [
        "Neither party shall be liable for delays caused by events outside its reasonable control.",
        "Force majeure events include acts of God, war, terrorism, and natural disasters.",
    ],
    "Warranty": [
        "Provider warrants that services will be performed in a professional manner.",
        "The products are warranted to be free from defects in materials for {n} months.",
    ],
}


def generate_synthetic():
    examples, label_to_idx = [], {l: i for i, l in enumerate(LABELS)}
    rng = np.random.default_rng(42)
    for label, phrases in TEMPLATES.items():
        idx = label_to_idx[label]
        for phrase in phrases:
            for _ in range(10):
                text = phrase.format(
                    n=int(rng.choice([30, 60, 90, 120])),
                    pct=round(rng.uniform(0.5, 2.0), 1),
                    amount=int(rng.choice([500000, 1000000, 5000000])),
                )
                examples.append({"text": text, "label": label, "label_idx": idx, "is_positive": True, "source": "synthetic", "split": "train"})
    for phrase in ["This agreement is entered into as of the effective date by and between the parties.", "IN WITNESS WHEREOF, the parties have executed this agreement.", "This agreement may be executed in counterparts, each of which shall be deemed an original."]:
        for _ in range(20):
            examples.append({"text": phrase, "label": "Entire Agreement", "label_idx": label_to_idx["Entire Agreement"], "is_positive": False, "source": "synthetic", "split": "train"})
    for text in ["This is a standard miscellaneous clause that does not fit any category.", "The parties acknowledge receipt of the exhibit attached hereto."]:
        for _ in range(20):
            for label in LABELS:
                examples.append({"text": text, "label": label, "label_idx": label_to_idx[label], "is_positive": False, "source": "synthetic", "split": "train"})
    logger.info(f"Generated {len(examples)} synthetic examples")
    return examples


def load_cuad():
    from datasets import load_dataset
    return load_dataset("cuad", trust_remote_code=True)


def load_ledgar():
    from datasets import load_dataset
    return load_dataset("lex_glue", "ledgar", trust_remote_code=True)


def cuad_to_examples(dataset):
    examples = []
    label_to_idx = {l: i for i, l in enumerate(LABELS)}
    for split in ["train", "test"]:
        if split not in dataset: continue
        for sample in dataset[split]:
            q = sample.get("question", "").lower()
            label = next((v for k, v in CUAD_MAP.items() if k in q), None)
            if label is None: continue
            has_ans = bool(sample.get("answers", {}) and any(t.strip() for t in sample["answers"].get("text", [])))
            examples.append({"text": sample.get("context", "")[:512], "label": label, "label_idx": label_to_idx[label], "is_positive": has_ans, "source": "cuad", "split": split})
    logger.info(f"Built {len(examples)} CUAD examples")
    return examples


def ledgar_to_examples(dataset):
    examples, label_to_idx = [], {l: i for i, l in enumerate(LABELS)}
    for split in ["train", "test", "validation"]:
        if split not in dataset: continue
        for sample in dataset[split]:
            ls = sample.get("label", ""); text = sample.get("text", sample.get("content", ""))
            if isinstance(ls, int): continue
            mapped = LEDGAR_MAP.get(ls)
            if mapped is None: continue
            examples.append({"text": text[:512], "label": mapped, "label_idx": label_to_idx[mapped], "is_positive": True, "source": "ledgar", "split": split if split != "validation" else "train"})
    logger.info(f"Built {len(examples)} LEDGAR examples")
    return examples


def build_dataset():
    all_examples = []
    if not USE_SYNTHETIC:
        for loader, converter in [(load_cuad, cuad_to_examples), (load_ledgar, ledgar_to_examples)]:
            try:
                all_examples.extend(converter(loader()))
            except Exception as e:
                logger.warning(f"Dataset error: {e}")
    all_examples.extend(generate_synthetic())
    train = [e for e in all_examples if e["split"] == "train"]
    test = [e for e in all_examples if e["split"] == "test"]
    if not test:
        rng = np.random.default_rng(42); idxs = rng.choice(len(train), size=max(1, len(train)//5), replace=False)
        test = [train[i] for i in idxs]; train = [e for i,e in enumerate(train) if i not in idxs]
    logger.info(f"Train: {len(train)}, Test: {len(test)}")
    return train, test


def train(train_examples, test_examples):
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback
    import torch
    from torch.utils.data import Dataset

    class ClauseDataset(Dataset):
        def __init__(self, examples, tokenizer, max_len=256):
            self.texts = [e["text"] for e in examples]
            self.labels = [e["label_idx"] for e in examples]
            self.tokenizer = tokenizer; self.max_len = max_len
        def __len__(self): return len(self.texts)
        def __getitem__(self, idx):
            enc = self.tokenizer(self.texts[idx], truncation=True, padding="max_length", max_length=self.max_len, return_tensors="pt")
            return {"input_ids": enc["input_ids"].squeeze(0), "attention_mask": enc["attention_mask"].squeeze(0), "labels": torch.tensor(self.labels[idx], dtype=torch.long)}

    import transformers.utils.import_utils as _iu
    _orig = getattr(_iu, "check_torch_load_is_safe", None)
    if _orig: _iu.check_torch_load_is_safe = lambda: None
    try:
        tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
        model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL, num_labels=len(LABELS), problem_type="single_label_classification")
    finally:
        if _orig: _iu.check_torch_load_is_safe = _orig

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logger.info(f"Device: {device}")
    if torch.cuda.is_available(): logger.info(f"GPU: {torch.cuda.get_device_name()}")

    train_dataset = ClauseDataset(train_examples, tokenizer)
    test_dataset = ClauseDataset(test_examples, tokenizer)

    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR, eval_strategy="epoch", save_strategy="epoch",
        learning_rate=LR, per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE * 2, num_train_epochs=EPOCHS,
        weight_decay=0.01, warmup_ratio=0.1,
        logging_dir=os.path.join(OUTPUT_DIR, "logs"), logging_steps=10,
        load_best_model_at_end=True, metric_for_best_model="eval_loss",
        greater_is_better=False, save_total_limit=2, dataloader_pin_memory=False,
        fp16=torch.cuda.is_available(),
    )

    trainer = Trainer(
        model=model, args=training_args,
        train_dataset=train_dataset, eval_dataset=test_dataset,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    logger.info("Starting training...")
    trainer.train()

    model_dir = os.path.join(OUTPUT_DIR, "final")
    model.save_pretrained(model_dir)
    tokenizer.save_pretrained(model_dir)
    with open(os.path.join(OUTPUT_DIR, "label_map.json"), "w") as f:
        json.dump({i: l for i, l in enumerate(LABELS)}, f, indent=2)
    eval_result = trainer.evaluate()
    logger.info(f"Evaluation: {eval_result}")
    with open(os.path.join(OUTPUT_DIR, "eval_results.json"), "w") as f:
        json.dump(eval_result, f, indent=2)
    logger.info(f"Model saved to {model_dir}")
    return model_dir

In [ ]:
train_examples, test_examples = build_dataset()
model_dir = train(train_examples, test_examples)
print(f"\nDONE! Model saved to {model_dir}")

In [ ]:
import shutil, os
shutil.make_archive("/content/clause_classifier_export", "zip", OUTPUT_DIR)
print("ZIP created at /content/clause_classifier_export.zip")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/normlens_models"
os.makedirs(DRIVE_DIR, exist_ok=True)
shutil.copy("/content/clause_classifier_export.zip", os.path.join(DRIVE_DIR, "clause_classifier_export.zip"))
print(f"Saved to Google Drive: {DRIVE_DIR}/clause_classifier_export.zip")
print("Download from Drive instead — much faster!")